# LAB : Spark MLlib

### Étape 0 : Initialisation de Spark

In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

spark = SparkSession.builder.appName("Lab4").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [2]:
print(spark.sparkContext._jvm.System.getProperty("java.version"))

17.0.12


### Étape 1 : Stocker le contenu de tous les fichiers du dossier tp4_data dans un DataFrame

In [5]:
import glob
import pandas as pd

files = glob.glob("C:/Users/User/Desktop/tp4_Spark_Ml/tp4_data/tp4_data/*.csv")
pdf = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
pdf["InvoiceDate"] = pd.to_datetime(pdf["InvoiceDate"])

df = spark.createDataFrame(pdf)
df.show(5)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|   17850.0|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|   17850.0|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
only showing top 5 rows



### Étape 2 : Afficher le schéma du DataFrame obtenu

In [6]:
df.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)



### Étape 3 : Remplir les valeurs manquantes (NaN) avec la valeur 0

In [7]:
df = df.na.fill(0)

### Étape 4 : Ajouter une nouvelle colonne "day_of_week" (jour de la semaine correspondant à "InvoiceDate")

In [8]:
df = df.withColumn("day_of_week", F.date_format(F.col("InvoiceDate"), "EEEE"))

df.select("InvoiceDate", "day_of_week").show(5)

+-------------------+-----------+
|        InvoiceDate|day_of_week|
+-------------------+-----------+
|2010-12-01 08:26:00|  Wednesday|
|2010-12-01 08:26:00|  Wednesday|
|2010-12-01 08:26:00|  Wednesday|
|2010-12-01 08:26:00|  Wednesday|
|2010-12-01 08:26:00|  Wednesday|
+-------------------+-----------+
only showing top 5 rows



### Étape 5 : Diviser les données en ensemble d'apprentissage (avant 2010-12-13) et ensemble de test (à partir de 2010-12-13)

In [9]:
train = df.where("InvoiceDate < '2010-12-13'")
test = df.where("InvoiceDate >= '2010-12-13'")

print(train.count(), test.count())

26732 18676


### Étape 6 : Créer un StringIndexer pour transformer les jours de la semaine en valeurs numériques

In [10]:
indexer = StringIndexer() \
    .setInputCol("day_of_week") \
    .setOutputCol("day_of_week_index") \
    .setHandleInvalid("keep")

### Étape 7 : Éviter l'ordre implicite entre les jours (samedi > lundi) : comment résoudre ce problème ?

In [11]:
encoder = OneHotEncoder() \
    .setInputCol("day_of_week_index") \
    .setOutputCol("day_of_week_encoded") 

# avec one hot encoder le probleme de 6 > 1 laisee croire que samdi est plus grand que lundi , mais avec one hot chaque jour deviant un vecteur ou un seule case vaut 1 et lesa autres vont a zero 

### Étape 8 : Créer un VectorAssembler avec UnitPrice, Quantity et day_of_week_encoded

In [12]:
vectorAssembler = VectorAssembler() \
    .setInputCols(["UnitPrice", "Quantity", "day_of_week_encoded"]) \
    .setOutputCol("features")

### Étape 9 : Créer un pipeline configuré avec les résultats des étapes 6, 7 et 8

In [13]:
pipeline = Pipeline().setStages([indexer, encoder, vectorAssembler])

### Étape 10 : Le StringIndexer doit connaître le nombre de valeurs uniques à indexer : comment résoudre ce problème ?

In [14]:
fittedPipeline = pipeline.fit(train)

### Étape 11 : Transformer les données de l'ensemble d'apprentissage en se basant sur les stages du pipeline

In [15]:
transformedTraining = fittedPipeline.transform(train)

transformedTraining.select("UnitPrice", "Quantity", "day_of_week", "day_of_week_index", "day_of_week_encoded", "features").show(5, truncate=False)

+---------+--------+-----------+-----------------+-------------------+--------------------------+
|UnitPrice|Quantity|day_of_week|day_of_week_index|day_of_week_encoded|features                  |
+---------+--------+-----------+-----------------+-------------------+--------------------------+
|2.55     |6       |Wednesday  |0.0              |(6,[0],[1.0])      |(8,[0,1,2],[2.55,6.0,1.0])|
|3.39     |6       |Wednesday  |0.0              |(6,[0],[1.0])      |(8,[0,1,2],[3.39,6.0,1.0])|
|2.75     |8       |Wednesday  |0.0              |(6,[0],[1.0])      |(8,[0,1,2],[2.75,8.0,1.0])|
|3.39     |6       |Wednesday  |0.0              |(6,[0],[1.0])      |(8,[0,1,2],[3.39,6.0,1.0])|
|3.39     |6       |Wednesday  |0.0              |(6,[0],[1.0])      |(8,[0,1,2],[3.39,6.0,1.0])|
+---------+--------+-----------+-----------------+-------------------+--------------------------+
only showing top 5 rows



### Étape 12 : Créer une instance de KMeans avec 20 clusters

In [16]:
kmeans = KMeans().setK(20).setSeed(1).setFeaturesCol("features")

### Étape 13 : Lancer l'apprentissage de KMeans sur l'ensemble obtenu à l'étape 11

In [17]:
kmModel = kmeans.fit(transformedTraining)

Py4JJavaError: An error occurred while calling o196.fit.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 8 in stage 15.0 failed 1 times, most recent failure: Lost task 8.0 in stage 15.0 (TID 63) (DESKTOP-0US7IJV.mshome.net executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:612)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:594)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:38)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:789)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:766)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:525)
	at org.apache.spark.InterruptibleIterator.hasNext(InterruptibleIterator.scala:37)
	

### Étape 14 : Effectuer des prédictions sur l'ensemble de test

In [ ]:
transformedTest = fittedPipeline.transform(test)
predictions = kmModel.transform(transformedTest)

predictions.select("UnitPrice", "Quantity", "day_of_week", "prediction").show(10)

### Étape 15 : Calculer le coefficient de silhouette

In [ ]:
evaluator = ClusteringEvaluator() \
    .setPredictionCol("prediction") \
    .setFeaturesCol("features") \
    .setMetricName("silhouette")

silhouette = evaluator.evaluate(predictions)
print(silhouette)